# AI Assignment by Subhayan Das

## Import libraries

In [1]:
import pygame
import random
import time
import sys
from collections import deque
import numpy as np
import pandas as pd
import os
from PIL import Image, ImageDraw

pygame 2.6.1 (SDL 2.28.4, Python 3.9.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Constants

In [2]:
# Constants
GRID_SIZE = 30
CELL_SIZE = 5
WINDOW_SIZE = CELL_SIZE * (2 * GRID_SIZE + 1)
BACKGROUND_COLOR = (255, 255, 255)
WALL_COLOR = (0, 0, 0)
PATH_COLOR = (255, 255, 255)
START_COLOR = (0, 255, 0)
END_COLOR = (255, 0, 0)
PATHFIND_COLOR = (0, 0, 255)
VISITED_COLOR = (255, 255, 0)

GAMMA = 0.95
THETA = 1e-5
REWARD_STEP = -1
REWARD_GOAL = 100

## Path Finding Functions

In [3]:
def calculate_metrics(maze, start, end, screen, traversal_order, explored_nodes, path, start_time):
    def path_complexity(maze):
        dead_ends = 0
        branching_points = 0
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        
        for r in range(1, len(maze) - 1, 2):
            for c in range(1, len(maze[0]) - 1, 2):
                if maze[r][c] == 0:
                    open_neighbors = 0
                    for dr, dc in directions:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
                            open_neighbors += 1
                    if open_neighbors == 0:
                        dead_ends += 1
                    elif open_neighbors > 2:
                        branching_points += 1
        return dead_ends, branching_points

    dead_ends, branching_points = path_complexity(maze)

    total_cells = len(maze) * len(maze[0])
    walls = sum(row.count(1) for row in maze)
    open_cells = total_cells - walls
    wall_density = walls / total_cells

    execution_time = time.time() - start_time

    final_path_length = len(path)

    memory_usage = sys.getsizeof(maze) + sys.getsizeof(path) + sys.getsizeof(explored_nodes)

    return {
        "Algorithm" : "MDP Value Iteration",
        "Grid Size" : GRID_SIZE,
        "Path Complexity (Entropy)": (dead_ends, branching_points),
        "Wall Density (Open Space Ratio)": wall_density,
        "Number of Explored Nodes (Search Cost)": len(explored_nodes),
        "Final Path Length": final_path_length,
        "Execution Time (Seconds)": execution_time,
        "Memory Usage (Bytes)": memory_usage
    }

In [4]:
def value_iteration(maze):
    rows, cols = len(maze), len(maze[0])
    values = np.zeros((rows, cols))
    policy = np.zeros((rows, cols, 2), dtype=int)
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    end = (rows - 2, cols - 2)
    
    while True:
        delta = 0
        for r in range(1, rows - 1):
            for c in range(1, cols - 1):
                if (r, c) == end or maze[r][c] == 1:
                    continue
                best_value = float('-inf')
                best_action = (0, 0)
                for dr, dc in directions:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == 0:
                        new_value = REWARD_STEP + GAMMA * values[nr][nc]
                        if new_value > best_value:
                            best_value = new_value
                            best_action = (dr, dc)
                delta = max(delta, abs(values[r][c] - best_value))
                values[r][c] = best_value
                policy[r][c] = best_action
        if delta < THETA:
            break
    return policy

In [5]:
def mdp_traversal(maze, screen):
    policy = value_iteration(maze)
    start = (1, 1)
    end = (len(maze) - 2, len(maze[0]) - 2)
    path = []
    traversal_order = []
    visited = set()
    r, c = start
    while (r, c) != end:
        path.append((r, c))
        traversal_order.append((r, c))
        visited.add((r, c))
        dr, dc = policy[r][c]
        r, c = r + dr, c + dc
        draw_maze(screen, maze, path, traversal_order)
        pygame.display.flip()
        pygame.time.delay(1)
    path.append(end)

    # Drawing the final path in blue
    draw_maze(screen, maze, path)
    pygame.display.flip()

    return path, traversal_order, visited

In [6]:
def draw_maze(screen, maze, path, visited_cells=[]):
    for r in range(len(maze)):
        for c in range(len(maze[0])):
            color = WALL_COLOR if maze[r][c] == 1 else PATH_COLOR
            pygame.draw.rect(screen, color, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in visited_cells:
        pygame.draw.rect(screen, VISITED_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in path:
        pygame.draw.rect(screen, PATHFIND_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    pygame.draw.rect(screen, START_COLOR, (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE))
    pygame.draw.rect(screen, END_COLOR, ((len(maze[0]) - 2) * CELL_SIZE, (len(maze) - 2) * CELL_SIZE, CELL_SIZE, CELL_SIZE))


In [7]:
def export_path(maze, path, start, end, filename):
    rows, cols = len(maze), len(maze[0])
    image = Image.new("RGB", (cols * CELL_SIZE, rows * CELL_SIZE), (255, 255, 255))
    draw = ImageDraw.Draw(image)

    for r in range(rows):
        for c in range(cols):
            if maze[r][c] == 1:  # Wall
                draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 0))

    for r, c in path:
        draw.rectangle([c * CELL_SIZE, r * CELL_SIZE, (c + 1) * CELL_SIZE, (r + 1) * CELL_SIZE], fill=(0, 0, 255))

    start_r, start_c = start
    draw.rectangle([start_c * CELL_SIZE, start_r * CELL_SIZE, (start_c + 1) * CELL_SIZE, (start_r + 1) * CELL_SIZE], fill=(0, 255, 0))

    end_r, end_c = end
    draw.rectangle([end_c * CELL_SIZE, end_r * CELL_SIZE, (end_c + 1) * CELL_SIZE, (end_r + 1) * CELL_SIZE], fill=(255, 0, 0))

    image.save(filename)
    print(f"Final path image saved as {filename}")

## Main Function

In [8]:
def main():
    pygame.init()

    screen_mdp = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))
    pygame.display.set_caption("MDP Value Iteration Maze Traversal")

    maze = np.load(f"./Mazes/{GRID_SIZE}.npy").tolist()

    start_time_mdp = time.time()
    mdp_path, mdp_traversal_order, mdp_visited = mdp_traversal(maze, screen_mdp)
    mdp_metrics = calculate_metrics(maze, (1, 1), (len(maze) - 2, len(maze[0]) - 2), screen_mdp, mdp_traversal_order, mdp_visited, mdp_path, start_time_mdp)

    print("MDP Value Iteration Metrics:")
    for metric, value in mdp_metrics.items():
        print(f"{metric}: {value}")

    # Saving the metrics:
    filename = 'results.csv'
    df = pd.DataFrame([mdp_metrics])

    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        df = pd.concat([existing_df, df], ignore_index=True)
    
    df.to_csv(filename, index=False)
    
    export_path(maze, mdp_path, (1, 1), (len(maze) - 2, len(maze[0]) - 2), f"./Paths/MDP_Val_{GRID_SIZE}.png")
    
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        pygame.display.flip()

    pygame.quit()

In [9]:

if __name__ == "__main__":
    main()

MDP Value Iteration Metrics:
Algorithm: MDP Value Iteration
Grid Size: 30
Path Complexity (Entropy): (0, 485)
Wall Density (Open Space Ratio): 0.44396667562483205
Number of Explored Nodes (Search Cost): 120
Final Path Length: 121
Execution Time (Seconds): 0.7812564373016357
Memory Usage (Bytes): 10032
Final path image saved as ./Paths/MDP_Val_30.png
